# AgentMind Tutorial: Getting Started with Multi-Agent Systems

Welcome to AgentMind! This tutorial will guide you through building your first multi-agent system.

## What You'll Learn

1. Creating your first agent
2. Setting up LLM providers
3. Building multi-agent collaboration
4. Adding tools to agents
5. Real-world examples

## Prerequisites

- Python 3.8+
- AgentMind installed (`pip install -e .`)
- Ollama installed (for local models) OR OpenAI API key

## Part 1: Your First Agent

Let's start by creating a simple agent.

In [ ]:
# Import required modules
from agentmind import Agent, AgentMind
from agentmind.llm import OllamaProvider
import asyncio

# For Jupyter notebooks, we need this helper
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# Create an LLM provider (using Ollama with local model)
llm = OllamaProvider(model="llama3.2")

# Create your first agent
agent = Agent(
    name="Assistant",
    role="helper",
    system_prompt="You are a helpful assistant who provides clear, concise answers."
)

print(f"Created agent: {agent.name}")
print(f"Role: {agent.role}")

## Part 2: Single Agent Interaction

Let's have a simple conversation with our agent.

In [ ]:
async def chat_with_agent():
    # Create AgentMind orchestrator
    mind = AgentMind(llm_provider=llm)
    mind.add_agent(agent)
    
    # Ask a question
    response = await mind.collaborate(
        "What are the benefits of using multi-agent systems?",
        max_rounds=1
    )
    
    return response

# Run the async function
response = await chat_with_agent()
print("Agent Response:")
print(response)

## Part 3: Multi-Agent Collaboration

Now let's create multiple specialized agents that work together.

In [ ]:
# Create specialized agents
researcher = Agent(
    name="Researcher",
    role="research",
    system_prompt="""You are a thorough researcher who gathers facts and information.
    Focus on finding accurate, relevant information and presenting it clearly."""
)

writer = Agent(
    name="Writer",
    role="writer",
    system_prompt="""You are a creative writer who crafts engaging content.
    Take research findings and turn them into compelling, well-structured text."""
)

editor = Agent(
    name="Editor",
    role="editor",
    system_prompt="""You are an editor who reviews and improves content.
    Check for clarity, accuracy, and readability. Suggest improvements."""
)

print("Created 3 specialized agents:")
print(f"1. {researcher.name} - {researcher.role}")
print(f"2. {writer.name} - {writer.role}")
print(f"3. {editor.name} - {editor.role}")

In [ ]:
async def multi_agent_collaboration():
    # Create new AgentMind instance
    mind = AgentMind(llm_provider=llm)
    
    # Add all agents
    mind.add_agent(researcher)
    mind.add_agent(writer)
    mind.add_agent(editor)
    
    # Give them a task
    task = """Write a short blog post (3 paragraphs) about the benefits of 
    using Python for data science. Make it informative and engaging."""
    
    print("Starting collaboration...\n")
    
    result = await mind.collaborate(task, max_rounds=3)
    
    return result

# Run the collaboration
result = await multi_agent_collaboration()
print("\n" + "="*60)
print("FINAL RESULT:")
print("="*60)
print(result)

## Part 4: Adding Tools to Agents

Tools give agents the ability to perform actions and access external data.

In [ ]:
from agentmind.tools import Tool
import json

# Create a simple calculator tool
class CalculatorTool(Tool):
    def __init__(self):
        super().__init__(
            name="calculator",
            description="Perform basic math operations: add, subtract, multiply, divide",
            parameters={
                "operation": {"type": "string", "description": "Operation: add, subtract, multiply, divide"},
                "a": {"type": "number", "description": "First number"},
                "b": {"type": "number", "description": "Second number"}
            }
        )
    
    async def execute(self, operation: str, a: float, b: float) -> str:
        operations = {
            "add": a + b,
            "subtract": a - b,
            "multiply": a * b,
            "divide": a / b if b != 0 else "Error: Division by zero"
        }
        
        result = operations.get(operation, "Error: Unknown operation")
        return json.dumps({"operation": operation, "result": result})

# Create a weather tool (simulated)
class WeatherTool(Tool):
    def __init__(self):
        super().__init__(
            name="get_weather",
            description="Get current weather for a city",
            parameters={
                "city": {"type": "string", "description": "City name"}
            }
        )
    
    async def execute(self, city: str) -> str:
        # Simulated weather data
        weather_data = {
            "city": city,
            "temperature": 72,
            "condition": "Partly Cloudy",
            "humidity": 65,
            "wind_speed": 10
        }
        return json.dumps(weather_data)

print("Created two tools: Calculator and Weather")

In [ ]:
# Create an agent with tools
assistant_with_tools = Agent(
    name="SmartAssistant",
    role="assistant",
    system_prompt="""You are a helpful assistant with access to tools.
    Use the calculator for math problems and weather tool for weather queries.""",
    tools=[CalculatorTool(), WeatherTool()]
)

async def test_tools():
    mind = AgentMind(llm_provider=llm)
    mind.add_agent(assistant_with_tools)
    
    # Test calculator
    result1 = await mind.collaborate(
        "What is 156 multiplied by 23?",
        max_rounds=1
    )
    print("Math Query Result:")
    print(result1)
    print("\n" + "-"*60 + "\n")
    
    # Test weather
    result2 = await mind.collaborate(
        "What's the weather like in San Francisco?",
        max_rounds=1
    )
    print("Weather Query Result:")
    print(result2)

await test_tools()

## Part 5: Real-World Example - Research Team

Let's build a practical research team that can analyze topics comprehensively.

In [ ]:
# Create a research team
topic_analyst = Agent(
    name="TopicAnalyst",
    role="analyst",
    system_prompt="""You analyze topics and break them down into key areas to research.
    Identify the most important aspects and questions to investigate."""
)

fact_finder = Agent(
    name="FactFinder",
    role="researcher",
    system_prompt="""You research facts and gather information on specific topics.
    Focus on accuracy and cite key points."""
)

synthesizer = Agent(
    name="Synthesizer",
    role="synthesizer",
    system_prompt="""You synthesize research findings into coherent summaries.
    Create clear, well-organized reports from multiple sources of information."""
)

async def research_topic(topic: str):
    mind = AgentMind(llm_provider=llm)
    mind.add_agent(topic_analyst)
    mind.add_agent(fact_finder)
    mind.add_agent(synthesizer)
    
    task = f"""Research and provide a comprehensive overview of: {topic}
    
    Steps:
    1. Analyze the topic and identify key areas to research
    2. Gather facts and information on each area
    3. Synthesize findings into a clear, organized summary
    """
    
    print(f"Researching: {topic}\n")
    print("="*60)
    
    result = await mind.collaborate(task, max_rounds=3)
    
    return result

# Research a topic
research_result = await research_topic("The impact of artificial intelligence on healthcare")
print("\nRESEARCH REPORT:")
print("="*60)
print(research_result)

## Part 6: Memory and Context

Agents can maintain conversation history and context.

In [ ]:
async def conversation_with_memory():
    mind = AgentMind(llm_provider=llm)
    
    conversational_agent = Agent(
        name="Conversationalist",
        role="assistant",
        system_prompt="You are a friendly assistant who remembers context from the conversation."
    )
    
    mind.add_agent(conversational_agent)
    
    # First message
    response1 = await mind.collaborate(
        "My name is Alice and I love Python programming.",
        max_rounds=1
    )
    print("Turn 1:")
    print(response1)
    print("\n" + "-"*60 + "\n")
    
    # Second message - agent should remember
    response2 = await mind.collaborate(
        "What's my name and what do I like?",
        max_rounds=1
    )
    print("Turn 2:")
    print(response2)

await conversation_with_memory()

## Part 7: Best Practices

Here are some tips for building effective multi-agent systems:

### 1. Clear Role Definition
Give each agent a specific, well-defined role and system prompt.

### 2. Appropriate Number of Agents
- Simple tasks: 1-2 agents
- Medium complexity: 3-4 agents
- Complex tasks: 5+ agents

### 3. Set Reasonable max_rounds
- Quick tasks: 1-2 rounds
- Standard tasks: 3-4 rounds
- Complex tasks: 5+ rounds

### 4. Use Tools Wisely
Give agents tools for:
- External data access
- Calculations
- API calls
- File operations

### 5. Monitor Performance
Track token usage, costs, and response times.

## Next Steps

Now that you've completed this tutorial, you can:

1. Explore more examples in the `examples/` directory
2. Read the [Architecture Guide](../ARCHITECTURE.md)
3. Check out [Integration Examples](../docs/INTEGRATIONS.md)
4. Build your own multi-agent application
5. Join the community on [GitHub Discussions](https://github.com/cym3118288-afk/AgentMind/discussions)

## Additional Resources

- [FAQ](../FAQ.md) - Common questions and answers
- [API Documentation](../API.md) - Complete API reference
- [Performance Guide](../PERFORMANCE.md) - Optimization tips
- [Troubleshooting](../TROUBLESHOOTING.md) - Common issues and solutions

Happy building with AgentMind!